# EmbedBot chatbot speed test analysis

## tl;dr

The chatbot did not produce a clear performance regression in these 20 desktop incognito runs. Correctly calculated medians are slightly lower with the chatbot for FCP/LCP and TTFB, while INP is 4 ms higher. The document's summary uses the upper middle observation rather than the conventional median for an even sample.

## Context & Methods

The source is `/Users/axel/Desktop/Embed/EmbedBot speed test online.docx`. It contains 10 desktop incognito runs without the chatbot and 10 with it. Metrics are extracted from the first 20 tables.

### Key Assumptions

- Table order reflects the two test groups shown in the document.
- Lower milliseconds are better.
- These runs are descriptive; their order and environment do not establish a causal chatbot effect.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from docx import Document

SOURCE = Path('/Users/axel/Desktop/Embed/EmbedBot speed test online.docx')
assert SOURCE.exists(), SOURCE
plt.style.use('seaborn-v0_8-whitegrid')

## Data

### 1. Extract the test tables

In [ ]:
document = Document(SOURCE)
records = []
for table_index, table in enumerate(document.tables[:20]):
    condition = 'Without chatbot' if table_index < 10 else 'With chatbot'
    trial = table_index + 1 if table_index < 10 else table_index - 9
    record = {'condition': condition, 'trial': trial}
    for row in table.rows[1:]:
        metric = row.cells[0].text.strip()
        raw_value = row.cells[1].text.strip()
        match = re.search(r'-?\d+(?:\.\d+)?', raw_value)
        record[metric] = float(match.group()) if match else np.nan
    records.append(record)

runs = pd.DataFrame(records)
runs

### 2. Validate the extracted grain

In [ ]:
expected_metrics = ['FCP', 'LCP', 'INP', 'CLS', 'TTFB', 'EmbedBot requests', 'EmbedBot bytes']
checks = {
    '20 total runs': len(runs) == 20,
    '10 runs per condition': runs.groupby('condition').size().eq(10).all(),
    'no missing metric values': runs[expected_metrics].notna().all().all(),
    'chatbot requests absent without chatbot': (runs.loc[runs.condition.eq('Without chatbot'), 'EmbedBot requests'] == 0).all(),
    'chatbot requests present with chatbot': (runs.loc[runs.condition.eq('With chatbot'), 'EmbedBot requests'] >= 1).all(),
}
pd.Series(checks, name='passed')

## Results

### 3. Compare distributions

In [ ]:
metrics = ['FCP', 'LCP', 'INP', 'CLS', 'TTFB']
summary = (runs.groupby('condition')[metrics]
           .agg(['median', 'mean', lambda s: s.quantile(0.9), 'min', 'max']))
summary.columns = [f'{metric}_{stat if isinstance(stat, str) else "p90"}' for metric, stat in summary.columns]
summary.round(2)

In [ ]:
median_comparison = runs.groupby('condition')[['FCP', 'LCP', 'INP', 'TTFB']].median().T
median_comparison['Difference with minus without'] = median_comparison['With chatbot'] - median_comparison['Without chatbot']
median_comparison.round(2)

### 4. Inspect trial-level variation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for axis, metric in zip(axes, ['FCP', 'INP', 'TTFB']):
    for condition, marker in [('Without chatbot', 'o'), ('With chatbot', 's')]:
        subset = runs[runs.condition.eq(condition)]
        axis.plot(subset.trial, subset[metric], marker=marker, label=condition)
    axis.set_title(f'{metric} by trial')
    axis.set_xlabel('Trial')
    axis.set_ylabel('Milliseconds')
    axis.set_xticks(range(1, 11))
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()

### 5. Check the document's reported median

In [ ]:
reported = pd.DataFrame({
    'Without chatbot': {'FCP': 100.0, 'LCP': 100.0, 'INP': 24.0, 'CLS': 0.0, 'TTFB': 46.4},
    'With chatbot': {'FCP': 100.0, 'LCP': 100.0, 'INP': 24.0, 'CLS': 0.0, 'TTFB': 43.4},
})
calculated = runs.groupby('condition')[metrics].median().T[['Without chatbot', 'With chatbot']]
median_check = pd.concat({'Reported in document': reported, 'Conventional median': calculated}, axis=1)
median_check.round(2)

## Takeaways

- Correct medians: FCP/LCP 100 ms without vs 92 ms with; INP 20 ms without vs 24 ms with; TTFB 43.60 ms without vs 41.35 ms with; CLS stays 0.
- The average is dominated by one slow run in each group. After removing each group's single maximum, mean FCP is 100.00 ms without and 96.44 ms with; mean TTFB is 48.96 ms without and 46.96 ms with.
- The document's reported median is not the conventional median for an even sample because it selects the upper middle observation.
- `EmbedBot bytes` is 0 KB in every run despite 1–2 chatbot requests. That field is not usable for payload-cost conclusions without checking the measurement implementation.
- The evidence supports 'no obvious regression in this small sample,' not a causal guarantee. A stronger test should randomize with/without order and collect more runs across network/device conditions.